# Figure 4, S13, S14 — Bayesian optimisation toward a target spectrum

* **S24-M01**: PCA-pretrained prior (`data/inputs/prior_kernel_pca_model_norm6_cosine_k4.pkl`) + normalised PCA cosine cost on the first 4 scores of the frozen basis `data/inputs/pca_model_norm6.npz`; 8 evaluations shown (GP snapshot `gp_model_iter_002`).
* **S28-M01**: raw-spectrum cosine cost, no prior; 15 evaluations shown (`gp_model_iter_009`).

Code: `{bo,bo_maps,fig04}.py` (ports of `paperAnalysis/BO/compare_S24_S28.ipynb` and `gp_projections_S24_S28.ipynb`). Costs are read from `costs.csv` (the platform's values); the plotted spectra are the SG-smoothed mean of the last max(5, 20 %) rows of each measurement. GP maps are 2-D slices through the best measured point (flow bounds = the run's own points + 12 %, colour scale per map).

⚑ The GP pickles were written by the platform with scikit-learn 1.3.0 and are loaded here with 1.2.2 (posterior mean checked against the stored state). ⚑ The text gives the target λmax as 601 nm; the target spectrum peaks at 607 nm.

## Analysis code
All the code of this figure is in this notebook.

### `paths`

In [ ]:
import sys
import types

from IPython.core.magic import register_cell_magic

MODULE_SOURCE = {}


@register_cell_magic
def module(line, cell):
    """`%%module a.b`: run this cell as the Python module a.b, so that the original code keeps its own namespaces."""
    name = line.strip()
    parts = name.split(".")
    for i in range(1, len(parts)):                              # parent packages
        pkg = ".".join(parts[:i])
        if pkg not in sys.modules:
            sys.modules[pkg] = types.ModuleType(pkg)
            sys.modules[pkg].__path__ = []
            if i > 1:
                setattr(sys.modules[".".join(parts[:i - 1])], parts[i - 1], sys.modules[pkg])
    mod = types.ModuleType(name)
    mod.__package__ = name.rpartition(".")[0]
    sys.modules[name] = mod
    setattr(sys.modules[mod.__package__], parts[-1], mod)
    MODULE_SOURCE[name] = cell
    exec(compile(cell, f"<{name}>", "exec"), mod.__dict__)


In [ ]:
%%module toofast_analysis.paths
"""Repository paths."""
import os
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "environment.yml").exists())   # notebook: no __file__
# raw/ can live elsewhere (e.g. an extracted Zenodo archive): set TOOFAST_RAW
RAW = Path(os.environ.get("TOOFAST_RAW", ROOT / "raw"))
PROCESSED = ROOT / "data" / "processed"
INPUTS = ROOT / "data" / "inputs"      # frozen artefacts used by the platform during the runs (versioned, read-only)
RESOURCE_DIR = INPUTS
FIGURES = ROOT / "figures"


In [ ]:
from toofast_analysis.paths import *

### `names`

In [ ]:
"""Names shown on the figures: the run of a path, and "S31-M01 sample_1" for a sample."""
import re


def new_name(old):
    name = str(old).replace("\\", "/").strip("/").split("/")[-1]
    return re.sub(r"_((?:sample|init|iter)_\d+)$", r" \1", name)


label = new_name


### `bo`

In [ ]:
"""Bayesian-optimisation runs (raw/<run>/): costs, measurements, target, saved GP snapshots.

Port of the I/O of paperAnalysis/BO/compare_S24_S28.ipynb and gp_projections_S24_S28.ipynb.
Run layout: costs.csv (one cost per line, evaluation order init_0..init_5 then iter_1..), measurement_spectra_<eval>.csv
(';'-separated time series, col 0 = timestamp), target_spectrum.csv, gaussian_processes/gp_model_iter_NNN.pkl.
"""
import os
import pickle

import numpy as np
import pandas as pd
from scipy.signal import savgol_filter


N_INIT = 6                         # init_0..init_5
STEADY_FRAC, STEADY_MINROWS = 0.20, 5
SAVGOL_WINDOW, SAVGOL_POLY = 21, 3


def run_dir(tag, raw=None):
    return (RAW if raw is None else raw) / tag


def ordered_measurement_files(tag, raw=None):
    d = run_dir(tag, raw)
    files = [d / f"measurement_spectra_init_{i}.csv" for i in range(N_INIT)]
    k = 1
    while (d / f"measurement_spectra_iter_{k}.csv").exists():
        files.append(d / f"measurement_spectra_iter_{k}.csv"); k += 1
    return files


def load_costs(tag, raw=None):
    c = pd.read_csv(run_dir(tag, raw) / "costs.csv", header=None).iloc[:, 0]
    return pd.to_numeric(c, errors="coerce").dropna().to_numpy()


def _wl_and_matrix(df):
    return df.columns[1:].astype(float).to_numpy(), df.iloc[:, 1:].apply(pd.to_numeric, errors="coerce").to_numpy()


def steady_spectrum(path):
    """Mean of the last max(5, 20 %) rows of a measurement (plotting only; the BO cost used the last row)."""
    wl, A = _wl_and_matrix(pd.read_csv(path, sep=";"))
    n = len(A)
    k = min(max(STEADY_MINROWS, int(np.ceil(n * STEADY_FRAC))), n)
    return wl, np.nanmean(A[-k:], axis=0)


def load_target(tag, raw=None):
    wl, A = _wl_and_matrix(pd.read_csv(run_dir(tag, raw) / "target_spectrum.csv", sep=";"))
    return wl, A[0]


def smooth(y):
    """Savitzky-Golay (21, 3), robust to NaN and short arrays."""
    y = np.asarray(y, dtype=float)
    finite = np.isfinite(y)
    if not finite.all() and finite.any():
        idx = np.arange(len(y)); y = np.interp(idx, idx[finite], y[finite])
    w = SAVGOL_WINDOW + (SAVGOL_WINDOW % 2 == 0)
    w = min(w, len(y) if len(y) % 2 == 1 else len(y) - 1)
    return y if w <= SAVGOL_POLY else savgol_filter(y, w, SAVGOL_POLY)


def normalize(y):
    peak = np.nanmax(y)
    return y / peak if peak and np.isfinite(peak) and peak != 0 else y


def load_run(tag, max_eval=None, raw=None):
    """Costs (truncated to `max_eval`), measurement files, best evaluation and its steady spectrum."""
    costs_full, files_full = load_costs(tag, raw), ordered_measurement_files(tag, raw)
    if len(costs_full) != len(files_full):
        raise ValueError(f"{tag}: {len(costs_full)} costs but {len(files_full)} measurement files")
    costs = costs_full[:max_eval] if max_eval else costs_full
    files = files_full[:len(costs)]
    bi = int(np.argmin(costs))
    wl, best = steady_spectrum(files[bi])
    return dict(costs=costs, files=files, best_idx=bi, best_cost=float(costs[bi]), wl=wl, best_spec=best)


def last_row(path):
    """Last raw row of a measurement / target CSV = the spectrum the platform scored."""
    df = pd.read_csv(path, sep=";")
    return df.columns[1:].astype(float).to_numpy(), df.iloc[-1, 1:].astype(float).to_numpy()


def load_pca_basis(path=None):
    """Frozen PCA basis of the PCA-cost runs (data/inputs/pca_model_norm6.npz, delivered as used, not refitted)."""
    pass
    z = np.load(INPUTS / "pca_model_norm6.npz" if path is None else path, allow_pickle=True)
    return dict(wl=z["wavelengths"], mean=z["mean"], comps=z["components"], normalize=str(z["normalize"]))


def pca_cosine_cost(wl, a, wl_t, a_t, basis, k):
    """Platform PCA cost: interpolate on the basis grid, max-normalise (if the basis says so), project, keep the
    first k scores, 1 - cosine(measured, target). Cross-check of costs.csv only (the figures read costs.csv)."""
    def score(w, y):
        y = np.interp(basis["wl"], w, y)
        if basis["normalize"] == "max":
            y = y / np.max(np.abs(y))
        return basis["comps"][:k] @ (y - basis["mean"])
    s, t = score(wl, a), score(wl_t, a_t)
    return float(1 - s @ t / (np.linalg.norm(s) * np.linalg.norm(t)))


def load_gp(tag, gp_iter, raw=None):
    """Saved GP snapshot after N_INIT + gp_iter evaluations (sklearn GaussianProcessRegressor, 4 features)."""
    path = run_dir(tag, raw) / "gaussian_processes" / f"gp_model_iter_{gp_iter:03d}.pkl"
    with open(path, "rb") as f:
        d = pickle.load(f)
    gp = d["gaussian_process"]
    flows = np.asarray(d["training_flows"], float)
    costs = np.asarray(d["training_costs"], float)
    ndim, ntot = int(gp.n_features_in_), flows.shape[1]
    reag = d.get("reagents")
    names = ([str(x) for x in list(reag)][:ntot] if isinstance(reag, (list, tuple, np.ndarray)) and len(list(reag)) >= ntot
             else [f"pump {i + 1}" for i in range(ntot)])
    bi = int(np.argmin(costs))
    return dict(tag=tag, gp=gp, flows=flows, costs=costs, ndim=ndim, ntot=ntot,
                qtot=float(np.round(flows.sum(1).mean(), 6)), names=names, best_idx=bi,
                best_full=flows[bi].copy(), best_x=flows[bi, :ndim].copy(), best_cost=float(costs[bi]),
                gp_file=os.path.basename(path))


### `bo_maps`

In [ ]:
"""2-D slices of a saved BO Gaussian process through the best measured point (Fig 4 right, S13, S14).

Port of paperAnalysis/BO/gp_projections_S24_S28.ipynb (cells 1, 3, 5, 9, 13), same code and parameters.
Settings of the run that produced the paper panels: SHARED_AXES = False (flow bounds = this run's own points + 12 %),
slack pump = Acetic Acid, colour scale adaptive per map (1st-99th percentile).
"""
from itertools import combinations

import matplotlib.patheffects as pe
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
from matplotlib.ticker import MultipleLocator, NullFormatter, ScalarFormatter


FONT = dict(base=14, suptitle=20, title=17, label=17, tick=15, legend=15, annot=15, num=20, num_grid=12, small=13)
FONT5 = dict(suptitle=26, title=21, label=22, tick=20, legend=19, annot=19, num=26, num_grid=16, small=18)
FIG5_PUB_SIZE = (7.8, 6.0)
FONT5_PUB = dict(label=19, tick=17, legend=16, annot=16, num=18)
MAP_STYLE = {"figure.dpi": 110, "savefig.dpi": 300, "font.size": FONT["base"], "axes.linewidth": 0.9, "axes.grid": False,
         "legend.frameon": False, "axes.titlesize": FONT["title"], "axes.labelsize": FONT["label"],
         "xtick.labelsize": FONT["tick"], "ytick.labelsize": FONT["tick"], "legend.fontsize": FONT["legend"],
         "figure.titlesize": FONT["suptitle"]}
SLACK_PUMP = "Acetic Acid"
FLOW_UNIT = "µL/s"
RATIO_MARGIN = 0.15


def save_map(fig, path):
    with plt.rc_context(MAP_STYLE):
        fig.savefig(path, bbox_inches="tight")


# ── slices ───────────────────────────────────────────────────────────────────
def dim_bounds(e, margin=0.12):
    X = e.get("bound_flows", e["flows"])
    lo, hi = X.min(0), X.max(0)
    span = np.where(hi > lo, hi - lo, 1.0)
    return np.maximum(0.0, lo - margin * span), hi + margin * span


def shared_norm(fields, lo_pct=1, hi_pct=99):
    allv = np.concatenate([np.asarray(f, float).ravel() for f in fields])
    lo, hi = np.nanpercentile(allv, lo_pct), np.nanpercentile(allv, hi_pct)
    if not np.isfinite(hi - lo) or (hi - lo) < 1e-9:
        lo, hi = lo - 0.5, hi + 0.5
    return Normalize(lo, hi)


def choose_slack(e, i, j):
    non = [k for k in range(e["ntot"]) if k not in (i, j)]
    if SLACK_PUMP in e["names"]:
        s = e["names"].index(SLACK_PUMP)
        if s in non:
            return s
    if e["ndim"] in non:
        return e["ndim"]
    gp_non = [k for k in non if k < e["ndim"]]
    rng = np.ptp(e["flows"][:, :e["ndim"]], axis=0)
    return max(gp_non, key=lambda k: rng[k])


def slice_2d(e, i, j, n=60):
    nd, qtot = e["ndim"], e["qtot"]
    lo, hi = dim_bounds(e)
    a = np.linspace(lo[i], hi[i], n); b = np.linspace(lo[j], hi[j], n)
    A, B = np.meshgrid(a, b)
    s = choose_slack(e, i, j)
    P = np.tile(e["best_full"], (A.size, 1))
    P[:, i] = A.ravel(); P[:, j] = B.ravel()
    P[:, s] = qtot - (P.sum(1) - P[:, s])
    feas = (P >= -1e-9).all(1)
    mu = np.full(A.size, np.nan); sig = np.full(A.size, np.nan)
    if feas.any():
        m, sd = e["gp"].predict(P[feas, :nd], return_std=True)
        mu[feas], sig[feas] = m, sd
    return a, b, mu.reshape(n, n), sig.reshape(n, n)


def ratio_slice(e, n=90):
    """(AgNO3/NaBH4 ratio, log) x Ascorbic slice through the best: AgNO3 + NaBH4 kept at the best's sum, PVP and
    Acetic Acid absorb the balance pro rata of their flows at the best."""
    nm = e["names"]
    ia, ib, ic, ip, iac = (nm.index(k) for k in ("AgNO3", "NaBH4", "Ascorbic Acid", "PVP", "Acetic Acid"))
    F, qtot = e["flows"], e["qtot"]
    lr = np.log10(F[:, ia] / F[:, ib])
    lo, hi = dim_bounds(e)
    x = np.logspace(lr.min() - RATIO_MARGIN, lr.max() + RATIO_MARGIN, n)
    y = np.linspace(lo[ic], hi[ic], n)
    X, Y = np.meshgrid(x, y)
    S = e["best_full"][ia] + e["best_full"][ib]
    P = np.tile(e["best_full"], (X.size, 1))
    P[:, ia] = S * X.ravel() / (1 + X.ravel())
    P[:, ib] = S / (1 + X.ravel())
    P[:, ic] = Y.ravel()
    rem = qtot - (P[:, ia] + P[:, ib] + P[:, ic])
    w = e["best_full"][ip] / (e["best_full"][ip] + e["best_full"][iac])
    P[:, ip], P[:, iac] = w * rem, (1 - w) * rem
    feas = (P >= -1e-9).all(1)
    mu = np.full(X.size, np.nan); sig = np.full(X.size, np.nan)
    if feas.any():
        m, sd = e["gp"].predict(P[feas, :e["ndim"]], return_std=True)
        mu[feas], sig[feas] = m, sd
    pts = np.c_[F[:, ia] / F[:, ib], F[:, ic]]
    return x, y, mu.reshape(n, n), sig.reshape(n, n), pts, S


# ── label / legend placement (minimal overlap with points and their numbers) ───
def _overlap(box, o):
    x0, y0, x1, y1 = box
    w = np.clip(np.minimum(x1, o[:, 2]) - np.maximum(x0, o[:, 0]), 0, None)
    h = np.clip(np.minimum(y1, o[:, 3]) - np.maximum(y0, o[:, 1]), 0, None)
    return float(np.sum(w * h))


def _obstacles(fig, ax, pts, texts, r_pt=14):
    rend = fig.canvas.get_renderer()
    xy = ax.transData.transform(pts)
    boxes = [np.c_[xy - r_pt, xy + r_pt]]
    for t in texts:
        bb = t.get_window_extent(rend)
        boxes.append(np.array([[bb.x0, bb.y0, bb.x1, bb.y1]]))
    return np.vstack(boxes)


def _out_of_axes(box, axbb, margin=4):
    x0, y0, x1, y1 = box
    return (max(0, axbb.x0 + margin - x0) + max(0, x1 - axbb.x1 + margin) +
            max(0, axbb.y0 + margin - y0) + max(0, y1 - axbb.y1 + margin))


def place_label(fig, ax, xy_data, text, obstacles, gaps=(12, 30, 50), **kw):
    rend = fig.canvas.get_renderer()
    ann = ax.annotate(text, xy_data, textcoords="offset points", xytext=(0, 0), ha="center", va="center",
                      arrowprops=dict(arrowstyle="-", color="white", lw=1.0, shrinkA=0, shrinkB=9), **kw)
    fig.canvas.draw()
    bb = ann.get_window_extent(rend)
    w, h = bb.width + 8, bb.height + 8
    cx, cy = ax.transData.transform(xy_data)
    axbb = ax.get_window_extent(rend)
    best = None
    for gi, gap in enumerate(gaps):
        for k, th in enumerate(np.deg2rad(np.r_[45:405:22.5])):
            dx = np.cos(th) * (w / 2 + gap); dy = np.sin(th) * (h / 2 + gap)
            box = (cx + dx - w / 2, cy + dy - h / 2, cx + dx + w / 2, cy + dy + h / 2)
            score = _overlap(box, obstacles) + 1e4 * _out_of_axes(box, axbb) + 150 * gi + k
            if best is None or score < best[0]:
                best = (score, dx, dy, box)
    px2pt = 72.0 / fig.dpi
    ann.xyann = (best[1] * px2pt, best[2] * px2pt)
    return ann, best[3]


def place_legend(fig, ax, obstacles, n=13, **kw):
    rend = fig.canvas.get_renderer()
    leg = ax.legend(loc="lower left", bbox_to_anchor=(0, 0), **kw)
    fig.canvas.draw()
    bb, axbb = leg.get_window_extent(rend), ax.get_window_extent(rend)
    w, h, m = bb.width, bb.height, 4
    best = None
    for fx in np.linspace(0, 1, n):
        for fy in np.linspace(0, 1, n):
            x0 = axbb.x0 + m + fx * (axbb.width - w - 2 * m)
            y0 = axbb.y0 + m + fy * (axbb.height - h - 2 * m)
            box = (x0, y0, x0 + w, y0 + h)
            corner = min(np.hypot(fx - cx_, fy - cy_) for cx_ in (0, 1) for cy_ in (0, 1))
            score = _overlap(box, obstacles) + 50 * corner + (0 if (fx, fy) == (1, 1) else 1)
            if best is None or score < best[0]:
                best = (score, x0, y0)
    leg.set_bbox_to_anchor(((best[1] - axbb.x0) / axbb.width, (best[2] - axbb.y0) / axbb.height),
                           transform=ax.transAxes)
    return leg


# ── figures ──────────────────────────────────────────────────────────────────
def plot_ratio(e, which):
    """Fig 4 right: GP mean ('mean') or sigma ('sigma') on the (AgNO3/NaBH4, Ascorbic) slice."""
    with plt.rc_context(MAP_STYLE):
        x, y, mu, sig, pts, S = ratio_slice(e)
        field = mu if which == "mean" else sig
        cmap = "viridis" if which == "mean" else "magma"
        norm = shared_norm([field])
        fig, ax = plt.subplots(figsize=FIG5_PUB_SIZE)
        pcm = ax.pcolormesh(x, y, field, cmap=cmap, norm=norm, shading="auto")
        if which == "mean":
            try:
                ax.contour(x, y, field, levels=7, colors="white", linewidths=0.6, alpha=0.65)
            except Exception:
                pass
            pt_face, pt_edge, num_color = "white", "k", "white"
        else:
            pt_face, pt_edge, num_color = "none", "cyan", "cyan"
        n_pts = len(pts)
        is_init = np.arange(n_pts) < N_INIT
        ax.scatter(pts[is_init, 0], pts[is_init, 1], s=70, marker="o", facecolor=pt_face, edgecolor=pt_edge,
                   linewidth=1.1, zorder=5, label="initialization")
        ax.scatter(pts[~is_init, 0], pts[~is_init, 1], s=80, marker="^", facecolor=pt_face, edgecolor=pt_edge,
                   linewidth=1.1, zorder=5, label="BO iteration")
        num_txt = [ax.annotate(str(k + 1), pts[k], textcoords="offset points", xytext=(5, 5),
                               fontsize=FONT5_PUB["num"], color=num_color, zorder=6,
                               path_effects=[pe.withStroke(linewidth=1.6, foreground="black")]) for k in range(n_pts)]
        bi = e["best_idx"]
        ax.scatter(*pts[bi], marker="*", s=380, color="red", edgecolor="k", linewidth=0.9, zorder=7,
                   label="best measured")
        ax.set_xscale("log")
        ax.set_xlim(x[0], x[-1]); ax.set_ylim(y[0], y[-1])
        ax.xaxis.set_major_formatter(ScalarFormatter()); ax.xaxis.set_minor_formatter(NullFormatter())
        ax.yaxis.set_major_locator(MultipleLocator(1.0)); ax.yaxis.set_minor_locator(MultipleLocator(0.5))
        ax.set_xlabel("AgNO3 / NaBH4 flow ratio", fontsize=FONT5_PUB["label"])
        ax.set_ylabel(f"Ascorbic Acid flow ({FLOW_UNIT})", fontsize=FONT5_PUB["label"])
        ax.tick_params(labelsize=FONT5_PUB["tick"])
        cb = fig.colorbar(pcm, ax=ax, shrink=0.9)
        cb.set_label("GP mean (cost)" if which == "mean" else "GP sigma", fontsize=FONT5_PUB["label"])
        cb.ax.tick_params(labelsize=FONT5_PUB["tick"])
        fig.tight_layout()
        fig.canvas.draw()
        obst = _obstacles(fig, ax, pts, num_txt)
        _, cost_box = place_label(fig, ax, tuple(pts[bi]), f"cost={e['best_cost']:.4f}", obst,
                                  fontsize=FONT5_PUB["annot"], zorder=8,
                                  bbox=dict(boxstyle="round,pad=0.25", fc="white", ec="none", alpha=0.8))
        place_legend(fig, ax, np.vstack([obst, np.array([cost_box])]), fontsize=FONT5_PUB["legend"],
                     frameon=True, facecolor="white", edgecolor="none", framealpha=0.75)
    return fig, dict(norm=(norm.vmin, norm.vmax), sum_ag_nabh4=S)


def plot_grid(e, which, label):
    """S13 / S14: GP mean or sigma on all 10 pump pairs (slices through the best measured point)."""
    with plt.rc_context(MAP_STYLE):
        nt, names, bi = e["ntot"], e["names"], e["best_idx"]
        Xp = e["flows"]; n_pts = len(Xp)
        pairs = list(combinations(range(nt), 2))
        data = {p: slice_2d(e, p[0], p[1]) for p in pairs}
        idx = 2 if which == "mean" else 3
        norm = shared_norm([data[p][idx] for p in pairs])
        cmap = "viridis" if which == "mean" else "magma"
        num_color = "white" if which == "mean" else "cyan"
        ncol = 4; nrow = int(np.ceil(len(pairs) / ncol))
        fig, axes = plt.subplots(nrow, ncol, figsize=(5.6 * ncol, 4.8 * nrow))
        axes = np.atleast_1d(axes).ravel()
        for ax, (i, j) in zip(axes, pairs):
            a, b, mu, sig = data[(i, j)]
            field = mu if which == "mean" else sig
            ax.pcolormesh(a, b, field, cmap=cmap, norm=norm, shading="auto")
            if which == "mean":
                try:
                    ax.contour(a, b, field, levels=6, colors="white", linewidths=0.5, alpha=0.6)
                except Exception:
                    pass
                ax.scatter(Xp[:, i], Xp[:, j], s=22, facecolor="white", edgecolor="k", linewidth=0.6, zorder=5)
            else:
                ax.scatter(Xp[:, i], Xp[:, j], s=22, facecolor="none", edgecolor="cyan", linewidth=0.8, zorder=5)
            for k in range(n_pts):
                ax.annotate(str(k + 1), (Xp[k, i], Xp[k, j]), textcoords="offset points", xytext=(3, 3),
                            fontsize=FONT5["num_grid"], color=num_color, zorder=6,
                            path_effects=[pe.withStroke(linewidth=1.2, foreground="black")])
            ax.scatter(Xp[bi, i], Xp[bi, j], marker="*", s=240, color="red", edgecolor="k", linewidth=0.8, zorder=7)
            ax.annotate(f"cost={e['best_cost']:.4f}", (Xp[bi, i], Xp[bi, j]), textcoords="offset points",
                        xytext=(9, 6), fontsize=FONT5["small"], zorder=8,
                        bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none", alpha=0.75))
            ax.set_xlim(a[0], a[-1]); ax.set_ylim(b[0], b[-1])
            ax.set_xlabel(f"{names[i]} ({FLOW_UNIT})", fontsize=FONT5["small"])
            ax.set_ylabel(f"{names[j]} ({FLOW_UNIT})", fontsize=FONT5["small"])
            ax.tick_params(labelsize=FONT5["small"])
        for ax in axes[len(pairs):]:
            ax.axis("off")
        ttl = "GP posterior mean" if which == "mean" else "GP posterior uncertainty"
        lbl = "GP posterior mean (cost, GP units)" if which == "mean" else "GP posterior uncertainty (sigma)"
        fig.subplots_adjust(left=0.05, right=0.91, top=0.93, bottom=0.06, wspace=0.3, hspace=0.35)
        cax = fig.add_axes([0.93, 0.15, 0.014, 0.7])
        cb = fig.colorbar(ScalarMappable(norm, cmap), cax=cax)
        cb.set_label(lbl, fontsize=FONT5["label"])
        cb.ax.tick_params(labelsize=FONT5["tick"])
        fig.suptitle(f"{label} — {ttl} (all pump pairs incl. NaBH4, {len(e['flows'])} evals, "
                     f"sliced through best measured point)", fontsize=FONT5["suptitle"])
    return fig, dict(norm=(norm.vmin, norm.vmax))


### `fig04`

In [ ]:
"""Figure 4 (and S13, S14) — BO toward the target spectrum: S24-M01 (PCA-pretrained prior + normalised PCA cosine
cost) vs S28-M01 (raw-spectrum cosine, no prior).

Panels: cost vs evaluation (compare_S24_S28.ipynb `fig1_cost_vs_iteration_*`), target vs best spectrum
(`fig3_{normalized,raw}_*`), GP mean on the (AgNO3/NaBH4, Ascorbic) slice (gp_projections_S24_S28.ipynb
`fig5_ratio_mean_*`); S13 / S14 = `fig5_SI_{mean,sigma}_*` (bo_maps.plot_grid).
Budgets as in the originals: S24-M01 8 evaluations (GP snapshot iter 2), S28-M01 15 evaluations (iter 9).
"""
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.ticker import AutoMinorLocator


EXPERIMENTS = {
    "S24-M01": dict(label="PCA-pretrained prior + normalized PCA cost", color="#1f77b4", max_eval=8, gp_iter=2,
                      kind="normalized", cost_ylabel="Cost\nNormalized PCA-based cosine similarity"),
    "S28-M01": dict(label="Raw-spectrum cosine (no prior)", color="#d62728", max_eval=15, gp_iter=9,
                    kind="raw", cost_ylabel="Cost\nRaw cosine similarity"),
}
STYLE = {"figure.dpi": 110, "savefig.dpi": 300, "font.size": 11, "axes.linewidth": 0.9, "axes.grid": True,
         "grid.alpha": 0.25, "grid.linewidth": 0.6, "legend.frameon": False}
FIG1_FONT = dict(label=15, title=13, tick=13, legend=13, annot=13)


def save(fig, path):
    with plt.rc_context(STYLE):
        fig.savefig(path, bbox_inches="tight")


def compute(raw=None):
    runs = {t: load_run(t, m["max_eval"], raw) for t, m in EXPERIMENTS.items()}
    targets = {t: load_target(t, raw) for t in EXPERIMENTS}
    gps = {t: load_gp(t, m["gp_iter"], raw) for t, m in EXPERIMENTS.items()}
    return dict(runs=runs, targets=targets, gps=gps)


def plot_cost(r, tag):
    meta, d = EXPERIMENTS[tag], r["runs"][tag]
    with plt.rc_context(STYLE):
        fig, ax = plt.subplots(figsize=(5.5, 5.5))
        c = d["costs"]; x = np.arange(1, len(c) + 1)
        ax.scatter(x, c, s=36, color=meta["color"], alpha=0.80, edgecolor="none", zorder=2,
                   label="individual evaluations")
        ax.plot(x, np.minimum.accumulate(c), color=meta["color"], lw=2.4, label="best so far", zorder=3)
        bi = d["best_idx"]
        ax.scatter([bi + 1], [c[bi]], s=150, marker="*", color=meta["color"], edgecolor="k", linewidth=0.8,
                   zorder=5, label="best evaluation")
        ax.axvspan(0.5, N_INIT + 0.5, color="grey", alpha=0.10, lw=0)
        ax.text(N_INIT / 2 + 0.5, ax.get_ylim()[1], "initialization", ha="center", va="top",
                fontsize=FIG1_FONT["annot"], color="grey")
        ax.set_yscale("log")
        ax.set_xlim(0, len(c) + 0.5)
        ax.set_xticks(range(1, len(c) + 1, 2))
        ax.set_xlabel("Iteration (evaluation)", fontsize=FIG1_FONT["label"])
        ax.set_ylabel(meta["cost_ylabel"], fontsize=FIG1_FONT["label"])
        ax.set_title(f"{meta['label']}\nBayesian optimization convergence", fontsize=FIG1_FONT["title"])
        ax.tick_params(labelsize=FIG1_FONT["tick"])
        ax.legend(fontsize=FIG1_FONT["legend"], loc="lower left")
        fig.tight_layout()
    return fig


def plot_best_spectrum(r, tag):
    """Target (dashed) vs best measured spectrum, SG-smoothed; normalised to the peak for the PCA run."""
    meta, d = EXPERIMENTS[tag], r["runs"][tag]
    with plt.rc_context(STYLE):
        fig, ax = plt.subplots(figsize=(5.5, 5.5))
        twl, tsp = r["targets"][tag]
        wl, sp = d["wl"], d["best_spec"]
        if meta["kind"] == "raw":
            t_y, y = smooth(tsp), smooth(sp)
            ylabel, subtitle = "Intensity (a.u.)", "raw intensity"
        else:
            t_y, y = normalize(smooth(tsp)), normalize(smooth(sp))
            ylabel, subtitle = "Normalized intensity", "normalized (shape)"
            ax.set_ylim(0, 1.05)
        ax.plot(twl, t_y, color="black", ls="--", lw=2.2, label="Target", zorder=1)
        ax.plot(wl, y, color=meta["color"], lw=1.9, alpha=0.9, label=f"Best (cost={d['best_cost']:.4f})")
        ax.set_xlabel("Wavelength (nm)", fontsize=14)
        ax.set_ylabel(ylabel, fontsize=14)
        ax.set_title(f"{meta['label']}\nTarget vs best measured ({subtitle})", fontsize=10.5)
        ax.xaxis.set_minor_locator(AutoMinorLocator())
        ax.legend(fontsize=14, loc="best")
        fig.tight_layout()
        ax.tick_params(labelsize=14)
        ax.legend(fontsize=14)
    return fig


def plot_gp_ratio(r, tag, which="mean"):
    return plot_ratio(r["gps"][tag], which)


def plot_gp_grid(r, tag, which="mean"):
    return plot_grid(r["gps"][tag], which, EXPERIMENTS[tag]["label"])


def save_gp(fig, path):
    save_map(fig, path)


def all_figures(r):
    """{original file stem: (figure, save function)} — the original names, for the regression check."""
    out = {}
    for tag, meta in EXPERIMENTS.items():
        out[f"fig1_cost_vs_iteration_{tag}"] = (plot_cost(r, tag), save)
        out[f"fig3_{meta['kind']}_{tag}"] = (plot_best_spectrum(r, tag), save)
        for which in ("mean", "sigma"):
            out[f"fig5_ratio_{which}_AgNaBH4_Ascorbic_{tag}"] = (plot_gp_ratio(r, tag, which)[0], save_gp)
            out[f"fig5_SI_{which}_{tag}"] = (plot_gp_grid(r, tag, which)[0], save_gp)
    return out


## Figure

In [ ]:
from IPython.display import Image, display
FIGURES.mkdir(exist_ok=True)
r = compute()
for tag, d in r['runs'].items():
    print(f"{label(tag)}: {len(d['costs'])} evaluations, best = #{d['best_idx'] + 1}, cost {d['best_cost']:.4f}")


## Figure 4 — cost vs evaluation, best vs target spectrum, GP mean on the (AgNO₃/NaBH₄, ascorbic acid) slice

In [ ]:
for tag in EXPERIMENTS:
    run = new_name(tag)                    # file names use the iterative run name (S24-M01, S28-M01)
    panels = {f'fig04_cost_{run}': (plot_cost(r, tag), save),
              f'fig04_spectrum_{run}': (plot_best_spectrum(r, tag), save),
              f'fig04_gp_mean_{run}': (plot_gp_ratio(r, tag, 'mean')[0], save_gp)}
    for name, (fig, saver) in panels.items():
        for ext in ('png', 'pdf'):
            saver(fig, FIGURES / f'{name}.{ext}')
        display(Image(str(FIGURES / f'{name}.png')))


## S13 (PCA run) and S14 (raw run) — GP posterior mean and σ on all pump pairs

In [ ]:
for tag, s in (('S24-M01', 'figS13'), ('S28-M01', 'figS14')):
    for which in ('mean', 'sigma'):
        fig, info = plot_gp_grid(r, tag, which)
        for ext in ('png', 'pdf'):
            save_gp(fig, FIGURES / f'{s}_{which}.{ext}')
        display(Image(str(FIGURES / f'{s}_{which}.png')))
